In [ ]:
import numpy as np
import nibabel as nib
from pathlib import Path
import matplotlib.pyplot as plt

# Check first 5 training cases
data_dir = Path('..') / 'BraTS2020_training_data' / 'MICCAI_BraTS2020_TrainingData'
patient_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])[:5]

print("Checking first 5 training cases...\n")

for patient_dir in patient_dirs:
    print(f"\n{patient_dir.name}:")
    
    # Load segmentation
    seg_file = list(patient_dir.glob('*_seg.nii*'))[0]
    mask = nib.load(str(seg_file)).get_fdata()
    
    # Check unique values
    unique, counts = np.unique(mask, return_counts=True)
    print(f"  Unique labels: {unique}")
    print(f"  Voxel counts: {dict(zip(unique, counts))}")
    
    # Calculate percentages
    total = mask.size
    for label, count in zip(unique, counts):
        pct = (count / total) * 100
        print(f"    Label {int(label)}: {pct:.2f}%")
    
    # Check if tumor exists
    tumor_voxels = (mask > 0).sum()
    print(f"  Tumor voxels: {tumor_voxels} ({(tumor_voxels/total)*100:.2f}%)")
    
    if tumor_voxels == 0:
        print("  ⚠️ WARNING: NO TUMOR IN THIS CASE!")

In [ ]:
# Visualize one case
patient_dir = patient_dirs[0]
print(f"\nVisualizing {patient_dir.name}")

# Load image and mask
flair = nib.load(str(list(patient_dir.glob('*_flair.nii*'))[0])).get_fdata()
mask = nib.load(str(list(patient_dir.glob('*_seg.nii*'))[0])).get_fdata()

# Plot middle slice
slice_idx = flair.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(flair[:, :, slice_idx], cmap='gray')
axes[0].set_title('FLAIR Image')
axes[0].axis('off')

axes[1].imshow(mask[:, :, slice_idx], cmap='jet', vmin=0, vmax=3)
axes[1].set_title('Segmentation Mask')
axes[1].axis('off')

axes[2].imshow(flair[:, :, slice_idx], cmap='gray')
axes[2].imshow(mask[:, :, slice_idx], cmap='jet', alpha=0.5, vmin=0, vmax=3)
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"\nSlice {slice_idx} tumor voxels: {(mask[:, :, slice_idx] > 0).sum()}")